## 0 · One Kick, One Question

> **Match day. 20 metres from goal. One free kick.**
>
> The ball draws one smooth curve, but a simulation does not need to know the whole curve in advance. It can build the flight from many tiny questions:
>
> 1. Where is the ball now?
> 2. How far does horizontal velocity carry it during a tiny time slice?
> 3. How far does vertical velocity carry it during that same slice?
> 4. How much does gravity change vertical velocity for the next slice?
> 5. Repeat.
>
> That is calculus in action: use change **at this instant** to predict the **next small step**, then let many small steps accumulate into a journey.

The model uses 28 equal time slices. A cyan bar records the next horizontal change, a gold bar records the next vertical change, and gravity prepares the vertical velocity used by the following step.

Start with the pattern: the vertical changes begin positive, shrink near the top, pass through zero, and become negative as the ball falls.

![Gravity-only free-kick trajectory assembled from repeated local updates, clearing the wall and reaching the target](images/gravity-only-trajectory.svg)

**Read the construction:**

- the cyan bar asks, “How far forward during this tiny moment?”
- the gold bar asks, “Will the next point be higher or lower?”
- each sampled point lies on the same gravity-only trajectory;
- the path clears the wall, stays inside the goal, and reaches the chosen target.

No single update knows the whole curve. The journey appears because the same local question is asked again and again.

# Mathematical Foundations for Machine Learning

## Follow the Match in Time

The mathematics will arrive in the same order as the kick.

### 1. At contact: set the initial velocity

The player sees the ball, wall, goal, and target. At contact, the foot gives the ball an initial direction and speed. An **arrow** records both: its direction says where the ball starts moving, and its length records the speed. Mathematics calls that arrow a **vector**.

Two arrows help us describe the shot:

- the **target vector** points directly from the ball to the intended finish point;
- the **kick vector** is the ball's initial velocity as it leaves the foot.

Without gravity, the kick could point directly along the target vector. In this model, gravity continuously pulls downward during the flight, so the kick must start above the direct target line. The player sets the initial velocity; gravity turns that initial condition into the curved path and final position.

We will first reason from rise, run, travel time, and gravity to the needed launch direction. Only then will we name the trigonometric relationships and compare the two arrows with a **dot product**.

### 2. After contact: predict the next instant

Once the ball is moving, its current velocity predicts one nearby point. This is the local-change idea behind a **derivative**.

### 3. During the flight: repeat

One nearby prediction is not a journey. Repeating and adding the tiny changes builds the full arc. This is the accumulation idea behind **integration**.

### 4. Across many kicks: allow variation

One exact kick follows one curve. Repeated kicks form a spread of possible outcomes, which introduces **probability**.

### 5. At Challenger Deep: improve a control

The probe uses a local derivative to decide how a tiny thrust change affects mission score. Repeated downhill parameter steps become **gradient descent**.

> **First see the physical question. Then give its mathematical tool a name.**

## Match Geometry Used by the Simulation

The free kick must satisfy visible constraints:

- launch speed: **20 m/s**;
- wall: **9.15 m** away and **1.8 m** high;
- goal line: **20 m** away with a **2.44 m** crossbar;
- back-net target: about **21.5 m** away and **1.10 m** high.

The ball has radius `0.11 m`, so its centre must clear the wall by more than the wall height alone and stay fully inside the goal frame.

> **Simplifying assumption:** after contact, gravity is the only force in the model. We deliberately ignore air resistance, wind, and spin effects such as topspin, dip, and curl.

The next code cell only gives these physical quantities short names. Part 1 will build the launch calculation from the picture before evaluating it.

In [5]:
# Dependencies
import subprocess, sys

# Only install packages that are not already importable in this environment
required = [("numpy", "numpy"), ("scipy", "scipy")]
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  done {pkg}")

import numpy as np
from scipy import stats

np.random.seed(42)
print("All dependencies ready")

# Physical constants for the free kick scenario
g = 9.81          # gravity changes downward velocity by 9.81 m/s each second
v0 = 20.0        # launch speed (m/s)
BALL_RADIUS = 0.11
WALL_X = 9.15    # wall position (m)
WALL_H = 1.8     # wall height (m)
GOAL_X = 20.0    # front of the goal (m)
CROSS_H = 2.44   # crossbar height (m)
NET_X = 21.5     # back of the net (m)
TARGET_H = 1.10  # intended ball-centre height at back-net impact (m)
GOAL_BOTTOM = BALL_RADIUS
GOAL_TOP = CROSS_H - BALL_RADIUS


def ball_state(x, theta_deg):
    """Return time, height, and velocity when the ball reaches horizontal position x."""
    theta = np.radians(theta_deg)
    vx = v0 * np.cos(theta)
    t = x / vx
    vy = v0 * np.sin(theta) - g * t
    y = v0 * np.sin(theta) * t - 0.5 * g * t**2
    return {"x": float(x), "t": float(t), "y": float(y), "vx": float(vx), "vy": float(vy)}


def ball_height(x, theta_deg):
    """Height of the ball centre at horizontal position x and launch angle theta (degrees)."""
    return ball_state(x, theta_deg)["y"]


print("\nFree kick setup:")
print(f"  Launch speed: {v0} m/s")
print(f"  Gravity changes downward velocity by {g} m/s each second")
print(f"  Wall at {WALL_X}m, ball centre must clear {WALL_H + BALL_RADIUS:.2f}m")
print(f"  Goal plane at {GOAL_X}m, legal centre window [{GOAL_BOTTOM:.2f}, {GOAL_TOP:.2f}]m")
print(f"  Back-net target at ({NET_X:.1f}m, {TARGET_H:.2f}m)")

  ok  numpy
  ok  scipy
All dependencies ready

Free kick setup:
  Launch speed: 20.0 m/s
  Gravity changes downward velocity by 9.81 m/s each second
  Wall at 9.15m, ball centre must clear 1.91m
  Goal plane at 20.0m, legal centre window [0.11, 2.33]m
  Back-net target at (21.5m, 1.10m)


---

## Part 1 — At Contact: Compare Two Arrows

### Begin with rise and run

The target is `21.5 m` forward and `1.10 m` above the ball. Picture a right triangle:

- **run:** travel `21.5 m` across the pitch;
- **rise:** finish `1.10 m` higher;
- **sloping side:** the direct line from the ball to the target.

The rise is tiny compared with the run, so the direct line must have a small angle. The ratio

$$\frac{\text{rise}}{\text{run}}=\frac{1.10}{21.5}\approx0.051$$

records that steepness. **Tangent** is simply the trigonometric name for this rise-to-run ratio:

$$\tan(\theta_{\text{target}})=\frac{1.10}{21.5}$$

Inverse tangent asks, “Which angle has this steepness?”

$$\theta_{\text{target}}=\tan^{-1}(0.051)\approx2.93^\circ$$

### Why the kick must point higher

If gravity did not act, the player could launch directly along that $2.93^\circ$ line. But the ball spends time in the air, and gravity pulls it downward throughout that time. The launch therefore needs an upward allowance.

First split the launch-speed arrow into two useful parts:

- the **horizontal part** carries the ball toward the goal;
- the **vertical part** lifts the ball while gravity pulls it back down.

For an arrow at angle $\theta$, cosine names its horizontal share and sine names its vertical share. Multiplying those shares by the launch speed $v_0$ gives:

$$v_x=v_0\cos\theta,\qquad v_{y,0}=v_0\sin\theta$$

Now reason through a flight lasting $t$ seconds:

1. Horizontal speed does not change, so horizontal distance is speed times time:
   $$x=v_0\cos\theta\,t$$
2. Without gravity, the vertical part would carry the ball upward by $v_0\sin\theta\,t$.
3. Gravity means vertical velocity changes downward by $g$ metres per second every second. That downward change grows steadily from `0` to $gt$, so its average over the interval is $gt/2$.
4. Average downward speed times time gives the height lost to gravity:
   $$\text{gravity drop}=\frac{gt}{2}\,t=\frac12gt^2$$

The actual height is therefore the upward carry minus the accumulated gravity drop:

$$y=v_0\sin\theta\,t-\frac12gt^2$$

We want height at a particular horizontal position rather than at a chosen time. The horizontal relation tells us how long reaching $x$ takes:

$$t=\frac{x}{v_0\cos\theta}$$

Substituting that travel time into the height relation gives the gravity-only path:

$$y(x)=x\tan\theta-\frac{g x^2}{2v_0^2\cos^2\theta}$$

Only now do we insert the shot values: $x=21.5$, $y=1.10$, $v_0=20$, and $g=9.81$. The lower launch angle that reaches the target is:

$$\theta_{\text{kick}}\approx19.11^\circ$$

The direct target line is $2.93^\circ$, but the launch is $19.11^\circ$ because it must supply enough initial upward motion for gravity to remove during the flight.

### Compare the two directions

Now place two unit-length vectors at the ball:

1. the **kick vector**, which gives the ball's initial direction;
2. the **target vector**, which points directly to the intended finish point.

Their directions differ by about $16.18^\circ$. We want one signed score for how much the initial direction aligns with the direct target line:

- strongly positive when the arrows point mostly the same way;
- zero when they are perpendicular;
- negative when they point in opposite directions.

![Dot product visualized as the signed shadow of the kick arrow along the direct target line](images/free-kick-vector-projection.svg)

The green segment is the kick arrow's forward **shadow** on the target line. The formal geometric word for this shadow is a **projection**: how much of one vector lies along another vector.

The signed projection length is the intuition behind the **dot product**.

#### Predict First

The launch direction is about $19.11^\circ$. The direct target line is about $2.93^\circ$.

What should the projection look like?

1. Almost as long as the full arrow
2. About half the arrow
3. Pointing backward

Make the visual prediction before calculating the score.

### Let the Shadow Become a Number

The kick and direct target line differ by about **16.18°**. In the diagram, the green shadow should still be almost as long as the kick arrow.

Suppose every arrow is scaled to length `1`:

- a full forward shadow would score `1`;
- this nearly full shadow scores about `0.960`;
- no forward shadow would score `0`;
- a full backward shadow would score `−1`.

The familiar geometry function that converts an angle gap into this shadow fraction is **cosine**:

- `cos(0°) = 1` — full agreement;
- `cos(90°) = 0` — sideways;
- `cos(180°) = −1` — opposite;
- `cos(16.18°) ≈ 0.960` — this kick and target.

Only now compress the idea:

$$\text{shadow score}=\cos(\text{angle between the arrows})$$

The mathematical name for the shadow score is the **dot product**. Using short vector names gives:

$$\mathbf{kick}\cdot\mathbf{target}=\cos(\text{angle between them})$$

The notation records the picture; it does not replace it.

<details>
<summary><strong>How the computer recovers the same shadow from coordinates</strong></summary>

The $19.11^\circ$ kick arrow is stored approximately as:

> horizontal part `0.945`, vertical part `0.327`

The $2.93^\circ$ target arrow is stored approximately as:

> horizontal part `0.999`, vertical part `0.051`

Compare matching directions and add their agreements:

$$0.945\times0.999+0.327\times0.051\approx0.960$$

That is the same shadow score predicted by the diagram.

Now name the storage pattern. A 2D vector keeps its two components together:

$$\mathbf{v}=[v_x,v_y]$$

For any two vectors, multiply matching components and add:

$$\mathbf{a}\cdot\mathbf{b}=a_xb_x+a_yb_y$$

The general formula grows directly from the numerical kick example.

</details>

### Read the Score Like a Player

| Arrow relationship | Shadow on the direct target line | Meaning |
| --- | --- | --- |
| Same direction | Long and forward | Most of the kick points along the target line. |
| Nearly aligned | Almost full length | A modest angle difference loses little forward agreement. |
| Sideways | Almost no length | The kick does not help along the target line. |
| Opposite | Points backward | The kick points away from the target line. |

For the $19.11^\circ$ kick and $2.93^\circ$ target line, expect a positive score close to `1`.

Only now do we need the code. It uses the two angles solved from the same target geometry and calculates the shadow score numerically.

In [6]:
# Part 1: solve the pictured geometry, then compare the two directions
from scipy.optimize import brentq

# The direct target angle comes from the rise-to-run ratio explained above.
target_line_deg = np.degrees(np.arctan2(TARGET_H, NET_X))

# Find the lower angle where predicted target height minus intended height is zero.
def target_height_error(angle_deg):
    return ball_height(NET_X, angle_deg) - TARGET_H


launch_angle_deg = brentq(target_height_error, 5.0, 45.0)
assert np.isclose(target_height_error(launch_angle_deg), 0.0)

kick_dir = np.array([
    np.cos(np.radians(launch_angle_deg)),
    np.sin(np.radians(launch_angle_deg)),
])
target_dir = np.array([
    np.cos(np.radians(target_line_deg)),
    np.sin(np.radians(target_line_deg)),
])

# For unit vectors, the dot product is the signed projection length.
shadow_score = np.dot(kick_dir, target_dir)
angle_between = np.degrees(np.arccos(np.clip(shadow_score, -1, 1)))

print(f"Kick direction ({launch_angle_deg:.2f}°):   {kick_dir.round(3)}")
print(f"Target direction ({target_line_deg:.2f}°): {target_dir.round(3)}")
print(f"Angle between arrows: {angle_between:.2f}°")
print(f"Forward-shadow score: {shadow_score:.4f} out of 1")
print()
print("Interpretation: most of the launch direction points along the direct target line.")

Kick direction (19.11°):   [0.945 0.327]
Target direction (2.93°): [0.999 0.051]
Angle between arrows: 16.18°
Forward-shadow score: 0.9604 out of 1

Interpretation: most of the launch direction points along the direct target line.


#### What the Number Confirms

The score is **0.9604**, close to the maximum value of `1`. The picture predicted this: the kick arrow casts a long forward shadow onto the direct target line.

The dot product has completed one job:

> **Compare the initial launch direction with the direct displacement to the target.**

It does not predict the later position by itself. Once the ball leaves the foot, gravity changes the vertical velocity, so the question becomes:

> **Given the current position and velocity, where is the next nearby point?**

That question leads to derivatives and the close-up step diagram in Part 2.

<details>
<summary><strong>Try a different launch direction</strong></summary>

```python
# Try 90° for a sideways launch, then 200° for a backward-pointing launch.
kick_angle = 90
target_angle = target_line_deg

kick = np.array([np.cos(np.radians(kick_angle)), np.sin(np.radians(kick_angle))])
target = np.array([np.cos(np.radians(target_angle)), np.sin(np.radians(target_angle))])
score = np.dot(kick, target)
print(f"kick={kick_angle}°, target={target_angle:.2f}° → shadow score={score:.4f}")
```

</details>

---

## Part 2 — After Contact: Build One Nearby Move

The ball has left the foot. The launch conditions are fixed. In this simplified model, only gravity changes the velocity. Now ask:

> **Given the ball's current position and velocity, where should the next nearby point be?**

![One exact gravity-only update showing the horizontal move, vertical move, and gravity correction at step 10](images/gravity-only-step.svg)

The diagram zooms in on step 10 of the solved trajectory:

1. the **cyan edge** records the forward change during one tiny time slice;
2. the **gold edge** records the net height change during that same slice;
3. the opposite corner becomes **NEXT**;
4. NEXT becomes the next NOW, and the calculation repeats.

The rectangle is a measurement scaffold. The ball does not physically travel sideways and then vertically; the two perpendicular edges split one nearby displacement into horizontal and vertical components.

Across the repeated steps:

- early gold edges rise strongly;
- gravity makes each later rise smaller;
- near the top, the gold edge almost vanishes;
- after the top, it points downward.

The ball reaches peak height when the vertical change passes through zero. That changing local move is the intuition behind a **derivative**.

### Should the peak sit directly above the wall?

That picture is tempting, but it cannot reach this goal under gravity alone. A gravity-only arc that starts at pitch level is symmetric around its peak: it takes the same horizontal distance to come back down to launch height as it took to rise.

If the peak were directly above the wall at `9.15 m`, the ball would return to launch height after another `9.15 m`, at `18.30 m`. That is before the `20 m` goal line. The shot would hit the ground before scoring.

The physically coherent version is for wall clearance to happen **near the top of the arc**, with the peak shortly afterward. Here the ball clears the wall at about `2.02 m`, then rises only another `0.16 m` before peaking at about `2.18 m`.

#### Predict First

What is the ball doing as it clears the wall?

1. Still rising quickly
2. Near the top, with only a small rise left
3. Already falling

Use the shrinking gold edge to make the physical prediction, then run the code cell.

### From One Step to the Update Rules

At the start of step 10, one time slice lasts about `0.041 s`. Before writing a formula, follow what happens during that short interval.

The ball already has upward velocity. If that vertical velocity stayed unchanged for the whole slice, it would carry the ball upward by:

> **upward carry ≈ +0.120 m**

Gravity is changing the vertical velocity throughout the slice. The value `9.81 m/s²` means that each second adds `9.81 m/s` of downward velocity. During only `0.041 s`, the downward change grows from `0` to about:

> **downward velocity gained ≈ 9.81 × 0.041 ≈ 0.398 m/s**

The change ramps evenly from `0` to `0.398 m/s`, so its average is half, about `0.199 m/s`. Over `0.041 s`, that average removes:

> **gravity drop ≈ 0.199 × 0.041 ≈ 0.008 m**

The net height change is therefore:

> **height change = upward carry − gravity drop ≈ 0.120 − 0.008 = +0.112 m**

Now compress the story. Let $\Delta t$ be one time slice and let $v_x$, $v_y$ be the current velocity components.

Horizontally, distance is speed times time:

$$\Delta x=v_x\Delta t$$

Vertically, current velocity supplies $v_y\Delta t$. Gravity's velocity change ramps from zero to $g\Delta t$, so its average is half of that. The accumulated drop is:

$$\text{gravity drop}=\frac12(g\Delta t)\Delta t=\frac12g(\Delta t)^2$$

Subtract that drop from the upward carry:

$$\Delta y=v_y\Delta t-\frac12g(\Delta t)^2$$

Gravity also reduces vertical velocity by $g\Delta t$ before the next slice:

$$v_{y,\text{next}}=v_y-g\Delta t$$

The next code cell repeats exactly this story 28 times.

In [8]:
# Part 2: repeat the gravity-only update derived above
theta_deg = launch_angle_deg
theta_rad = np.radians(theta_deg)
step_count = 28

velocity_x = v0 * np.cos(theta_rad)
position_x = 0.0
position_y = 0.0
velocity_y = v0 * np.sin(theta_rad)
flight_time = NET_X / velocity_x
delta_t = flight_time / step_count

step_rows = []
for step_index in range(step_count):
    delta_x = velocity_x * delta_t
    delta_y = velocity_y * delta_t - 0.5 * g * delta_t**2
    next_velocity_y = velocity_y - g * delta_t

    if step_index in {0, 9, 18, 27}:
        step_rows.append((step_index + 1, position_x, position_y, delta_x, delta_y, next_velocity_y))

    position_x += delta_x
    position_y += delta_y
    velocity_y = next_velocity_y

# Compare the accumulated small steps with the direct projectile formula and target.
net_state = ball_state(NET_X, theta_deg)
step_error = np.hypot(position_x - net_state["x"], position_y - net_state["y"])
target_error = abs(net_state["y"] - TARGET_H)

print(f"28 slices across {flight_time:.3f}s give Δt={delta_t:.3f}s\n")
print(" step   start (x, y)       Δx        Δy       next vy")
for step_number, start_x, start_y, change_x, change_y, next_vy in step_rows:
    print(
        f" {step_number:>2d}    ({start_x:>5.2f}, {start_y:>4.2f})   "
        f"{change_x:+.3f}m   {change_y:+.3f}m   {next_vy:+.3f}m/s"
    )

print(f"\nAccumulated endpoint: ({position_x:.6f}, {position_y:.6f}) m")
print(f"Direct-formula endpoint: ({net_state['x']:.6f}, {net_state['y']:.6f}) m")
print(f"Intended target: ({NET_X:.6f}, {TARGET_H:.6f}) m")
print(f"Endpoint disagreement: {step_error:.2e} m")
print(f"Target-height error: {target_error:.2e} m")
assert step_error < 1e-10
assert target_error < 1e-10

x_peak_analytic = v0**2 * np.sin(2 * theta_rad) / (2 * g)
peak_state = ball_state(x_peak_analytic, theta_deg)
wall_state = ball_state(WALL_X, theta_deg)
height_left_after_wall = peak_state["y"] - wall_state["y"]
print(f"\nWall clearance: x={WALL_X:.2f}m, y={wall_state['y']:.2f}m, vy={wall_state['vy']:+.2f}m/s")
print(f"Turning point: x={x_peak_analytic:.2f}m, y={peak_state['y']:.2f}m, vy={peak_state['vy']:+.2e}m/s")
print(f"Height still to rise after the wall: {height_left_after_wall:.2f}m")
print("Prediction check: answer 2 — wall clearance happens near the top of the arc.")

28 slices across 1.138s give Δt=0.041s

 step   start (x, y)       Δx        Δy       next vy
  1    ( 0.00, 0.00)   +0.768m   +0.258m   +6.149m/s
 10    ( 6.91, 1.74)   +0.768m   +0.112m   +2.561m/s
 19    (13.82, 2.16)   +0.768m   -0.034m   -1.026m/s
 28    (20.73, 1.28)   +0.768m   -0.179m   -4.613m/s

Accumulated endpoint: (21.500000, 1.100000) m
Direct-formula endpoint: (21.500000, 1.100000) m
Intended target: (21.500000, 1.100000) m
Endpoint disagreement: 7.16e-15 m
Target-height error: 1.33e-15 m

Wall clearance: x=9.15m, y=2.02m, vy=+1.80m/s
Turning point: x=12.61m, y=2.18m, vy=-8.88e-16m/s
Height still to rise after the wall: 0.16m
Prediction check: answer 2 — wall clearance happens near the top of the arc.


#### Prediction Check — Wall Clearance Happens Near the Peak

Prediction **2** is correct. At the wall:

- the ball's centre is about **2.02 m** high, enough to clear the required **1.91 m**;
- vertical velocity has slowed to about **+1.80 m/s**, so the ball is still rising but much less quickly;
- only about **0.16 m** of rise remains.

The ball then reaches its **2.18 m** peak at about **12.61 m** before beginning to fall.

So the wall and peak correspond in **height and phase of motion**, even though they cannot share the exact horizontal position. Wall clearance occurs in the near-flat top region of the arc.

### Repeat the Same Nearby Question

One rectangle gives one next point. To build a journey, make that endpoint the new **NOW** and ask again.

![Animation showing 4, 8, 16, and 28 repeated local predictions becoming a progressively smoother free-kick curve](images/calculus-accumulation.gif)

Read the stages as a progression:

- **4 large steps:** the component staircase is visibly angular.
- **8 steps:** more rectangles follow the bend.
- **16 steps:** the staircase hugs the curve.
- **28 tiny steps:** the rectangular construction looks almost smooth.

The code has not introduced a new rule. It applies the one-step story repeatedly:

1. current velocity carries the ball during a short time;
2. gravity reduces vertical velocity during that time;
3. the resulting endpoint becomes the next starting point.

One local update uses the ball's current rate of change. That is the role of a **derivative**. Adding all the local position changes builds the journey. That is the role of **integration**.

Because gravity is constant in this simplified model, every interval can include its exact average effect rather than using only a tangent-line guess.

> **Read the current rate → let gravity change it → update position and velocity → repeat.**

#### Two Calculus Moves

The animations separate one continuous idea into two jobs:

1. **Read the present:** use the ball's current motion to predict one nearby point.
2. **Build the journey:** make that point the new starting point and repeat.

No single step contains a blueprint of the full arc. It only needs the current position, current velocity, constant gravity, and one short time slice.

> **Model boundary:** real football also includes air resistance, wind, and spin effects such as topspin, dip, and curl. We ignore all of them for simplicity. After contact, gravity is the only force in this model.

That leaves one repeated pattern:

> **read the current state → apply gravity → update → repeat**

A useful next question is therefore: how do we carry several pieces of state—position, velocity, time, and acceleration—together? Vectors collect them; matrices transform them.

---

## Part 3 — One Recipe, Then Many Recipes

The flight calculation carries several related values together. For a smaller example, keep only:

> angle signal `0.4`, speed signal `0.8`

Putting them in a fixed order creates one input vector:

> **input vector = [angle signal, speed signal] = [0.4, 0.8]**

Now suppose we want two internal answers:

1. a clearance signal;
2. a target-height signal.

One weighted recipe can produce one answer:

> multiply each input by its relevance, add the pieces, then add a fixed offset

That fixed offset is called a **bias**. It lets a recipe start above or below zero instead of being forced through zero.

The weighted part is exactly the dot-product move from Part 1, now used as a prediction recipe rather than an alignment score.

To produce both answers, stack two recipes:

![Matrix multiplication built from two stacked dot-product recipes applied to an angle-and-speed state vector](images/free-kick-matrix-recipes.svg)

This stack is called a **matrix**:

- one row stores one recipe;
- each column follows one input through all recipes;
- all row answers form the output vector.

#### Predict First

If both recipes use the angle input, what happens when only angle changes?

1. Only the first output can change
2. Every output connected to angle can change
3. Neither output changes

Use the diagram and the row calculations below before opening the compact notation.

> **Connection to Part 1:** a dot product runs one row. Matrix multiplication runs every row against the same input vector.

> **Connection to the kick:** angle and speed enter together. Different rows can listen to them differently, so one shared state can produce several useful signals.

In [ ]:
# Part 3: stack two dot-product recipes into one matrix
features = np.array([0.4, 0.8])  # normalized [angle signal, speed signal]

# Each row is one dot-product recipe applied to the same input vector.
W = np.array([
    [2.0, 0.5],   # row 1 listens strongly to angle
    [-0.5, 1.5],  # row 2 listens strongly to speed
])
b = np.array([0.1, -0.2])

output = W @ features + b

print(f"Input vector [angle, speed]: {features}")
print("\nRun each row as one dot product:")
for row_index, (row, bias, result) in enumerate(zip(W, b, output), start=1):
    angle_part, speed_part = row * features
    print(
        f"  row {row_index}: angle {angle_part:+.2f}, speed {speed_part:+.2f}, "
        f"bias {bias:+.2f} → result {result:.2f}"
    )

print(f"\nStacked matrix result: {output.round(3)}")
print("One row gives one output; two rows give an output vector of length two.")

### Read the Rows Before the Matrix

The input vector is:

- angle signal = `0.4`;
- speed signal = `0.8`.

Each row asks its own weighted question:

| Recipe | Angle contribution | Speed contribution | Bias | Result |
| --- | ---: | ---: | ---: | ---: |
| Row 1 | `2 × 0.4 = 0.8` | `0.5 × 0.8 = 0.4` | `+0.1` | **1.3** |
| Row 2 | `−0.5 × 0.4 = −0.2` | `1.5 × 0.8 = 1.2` | `−0.2` | **0.8** |

Read one row at a time:

- Row 1 listens strongly to angle and a little to speed.
- Row 2 treats angle as a small brake and listens strongly to speed.
- The bias shifts the final answer after the dot product.

In a real model, outputs could be calibrated quantities such as expected clearance or scoring confidence. Here `1.3` and `0.8` are deliberately uncalibrated internal signals: they exist only to expose how information flows from one input vector through several recipes.

Now read by columns:

- the first column contains every route taken by angle;
- the second column contains every route taken by speed.

> **Rows build outputs. Columns trace inputs. The matrix stores both views at once.**

#### Let the Rows Become Matrix Notation

The code has already run the concrete recipes:

- Row 1 turns `[0.4, 0.8]` into `1.3`.
- Row 2 turns `[0.4, 0.8]` into `0.8`.
- Keeping both answers in order gives `[1.3, 0.8]`.

The conceptual ladder is:

1. **Number:** one measured value.
2. **Vector:** several related values kept in order.
3. **Dot product:** one weighted recipe applied to a vector.
4. **Matrix:** several weighted recipes stacked together.

Now assign compact names:

- $\mathbf{x}$ = input vector `[0.4, 0.8]`;
- $W$ = stack of row recipes;
- $\mathbf{b}$ = one offset for each row;
- $\mathbf{y}$ = output vector `[1.3, 0.8]`.

The whole calculation becomes:

$$\mathbf{y}=W\mathbf{x}+\mathbf{b}$$

Read it as a sentence, not a spell:

> **run every row of $W$ against the same input $\mathbf{x}$, then add each row's offset**

Prediction (2) is correct: because angle appears in both rows, changing angle can change both outputs.

<details>
<summary><strong>Try changing one column</strong></summary>

```python
W_test = np.array([
    [3.0, 0.1],  # row 1 listens mostly to angle
    [0.1, 3.0],  # row 2 listens mostly to speed
])

normal = np.array([0.4, 0.8])
more_angle = np.array([0.8, 0.8])

print("normal:    ", W_test @ normal)
print("more angle:", W_test @ more_angle)
print("change:    ", W_test @ more_angle - W_test @ normal)
```

The first output changes much more because its row listens strongly to the angle column.

</details>

So far each kick used exact inputs. Real players vary from attempt to attempt. That leads to probability.

---

## Part 4 — One Kick Is Not a Cloud of Attempts

So far the same angle and speed always produce the same curve. That describes a perfectly repeatable kick.

A real player is not perfectly repeatable. Ask for 20° one hundred times and the actual kicks form a cloud around 20°:

- many land close to the intention;
- some are a little high or low;
- a few are farther away.

That cloud is a **probability distribution**.

The legal scoring angles span roughly **18.44° to 21.86°**. Compare two aims:

- **19.11°** sends one exact kick to the chosen back-net point;
- **20.15°** centres the whole cloud inside the legal window.

The visual question comes first:

> **Which centre leaves more of the cloud between the two legal boundaries?**

#### Predict First

Suppose the typical miss is about 3°. Which aim scores more often over 100 attempts?

1. 19.11°, because it is best for one exact kick
2. 20.15°, because it centres the whole cloud inside the legal window
3. They must be identical

### Let the Cloud Become Basic Probability Notation

Give the uncertain executed angle a name: $\Theta$.

The statement

> executed angle lands between 18.44° and 21.86°

becomes

$$18.44°<\Theta<21.86°$$

Placing $P(\cdot)$ around an event means “the probability of this event”:

$$P(18.44°<\Theta<21.86°)$$

This expression means:

> **the fraction of the kick cloud that falls inside the legal window**

For a cloud centred at 20.15° with a typical spread of 3°, the model estimates a fraction of about `0.431`: roughly 43 scores per 100 attempts.

A bell-shaped **Gaussian distribution** is used as a convenient model for the execution cloud. It is an assumption, not a physical law of football. Real kick errors need not form a perfect bell curve.

<details>
<summary><strong>Optional advanced connection: turn probability into a surprise score</strong></summary>

A likely event should have a small surprise penalty; an unlikely event should have a large one.

- chance `0.9` → small surprise;
- chance `0.1` → large surprise.

Why use a logarithm? Probabilities from several independent events multiply, while their logarithms add. Addition is easier to accumulate and optimize across many observations.

Name the chance $p$. The surprise score is:

$$\text{surprise}=-\log(p)$$

The minus sign makes small probabilities produce large positive penalties. When $p$ is the probability assigned to the observed outcome, this score is called **negative log-likelihood**.

</details>

> **Keep the objects separate:** one exact input produces one trajectory. Repeated imperfect inputs produce a cloud of trajectories. Probability describes the cloud rather than pretending every attempt is identical.

In [4]:
# Part 4: compare one point-target aim with a reliable repeated-attempt aim
# Keep angles that clear the wall and cross fully inside the goal frame
scoreable_angles = [
    angle for angle in np.linspace(5, 60, 20_000)
    if (
        ball_height(WALL_X, angle) > WALL_H + BALL_RADIUS
        and GOAL_BOTTOM < ball_height(GOAL_X, angle) < GOAL_TOP
    )
]

if not scoreable_angles:
    raise RuntimeError("No launch angle satisfies the physical constraints")

theta_lo = min(scoreable_angles)
theta_hi = max(scoreable_angles)
point_target_aim = launch_angle_deg
cluster_centered_aim = (theta_lo + theta_hi) / 2
sigma = 3.0


def probability_of_scoring(intended_angle, execution_sigma=sigma):
    distribution = stats.norm(loc=intended_angle, scale=execution_sigma)
    return distribution.cdf(theta_hi) - distribution.cdf(theta_lo)


probability_point_target = probability_of_scoring(point_target_aim)
probability_cluster_centered = probability_of_scoring(cluster_centered_aim)
nll_point_target = -np.log(probability_point_target)
nll_cluster_centered = -np.log(probability_cluster_centered)

print(f"Legal angle window: [{theta_lo:.2f}°, {theta_hi:.2f}°]")
print(f"Typical execution spread: {sigma:.1f}°\n")
print("ONE EXACT INPUT VERSUS A DISTRIBUTION")
print(
    f"  Aim at one chosen point: {point_target_aim:5.2f}°  "
    f"expected scores per 100={100 * probability_point_target:4.1f}"
)
print(
    f"  Center the whole cluster: {cluster_centered_aim:5.2f}°  "
    f"expected scores per 100={100 * probability_cluster_centered:4.1f}"
)
print(f"  Extra expected scores per 100: {100 * (probability_cluster_centered - probability_point_target):+.1f}")
print(f"  Surprise penalty: {nll_point_target:.3f} → {nll_cluster_centered:.3f}")
print("\nPrediction check: answer (b) — centering the cluster leaves more room for imperfect kicks.\n")

print("Consistency check at the cluster-centered aim:")
for spread in [1.0, 2.0, 3.0, 5.0, 8.0, 10.0]:
    probability = probability_of_scoring(cluster_centered_aim, spread)
    print(f"  typical spread={spread:4.1f}°: expected scores per 100={100 * probability:4.1f}")

Legal angle window: [18.44°, 21.86°]
Typical execution spread: 3.0°

ONE EXACT INPUT VERSUS A DISTRIBUTION
  Aim at one chosen point: 19.11°  expected scores per 100=40.9
  Center the whole cluster: 20.15°  expected scores per 100=43.1
  Extra expected scores per 100: +2.3
  Surprise penalty: 0.895 → 0.841

Prediction check: answer (b) — centering the cluster leaves more room for imperfect kicks.

Consistency check at the cluster-centered aim:
  typical spread= 1.0°: expected scores per 100=91.3
  typical spread= 2.0°: expected scores per 100=60.8
  typical spread= 3.0°: expected scores per 100=43.1
  typical spread= 5.0°: expected scores per 100=26.8
  typical spread= 8.0°: expected scores per 100=16.9
  typical spread=10.0°: expected scores per 100=13.6


### Imagine 100 repeated kicks

With a 3° typical spread, the two aims behave differently over many attempts:

| Strategy | Intended angle | Expected scores out of 100 | What it describes |
| --- | ---: | ---: | --- |
| Hit one back-net point precisely | 19.11° | About **41** | One exact gravity-only trajectory |
| Center varied attempts in the legal window | 20.15° | About **43** | Probability of any legal score |

The gain is only about two extra goals per 100 kicks here, but the principle is durable:

> **A single predicted value and a distribution of possible values answer different questions.**

Consistency matters even more than the one-degree change in aim:

| Typical spread around the aim | Rough scores out of 100 at the centered aim | What the cluster looks like |
| ---: | ---: | --- |
| 1° | **91** | Tight and consistent |
| 2° | **61** | Wider, but most attempts remain near the aim |
| 3° | **43** | Many attempts spill outside the narrow legal window |
| 5° | **27** | Broad scatter |
| 8° | **17** | Very broad scatter |

Moving the **aim** shifts the centre of the distribution. Improving **consistency** tightens it. Probability lets us reason about both without pretending every repeated attempt is identical.

#### What just happened

The two questions preferred different reference angles:

- **19.11°** sends one exact kick to the chosen back-net point,
- **20.15°** gives a spread of imperfect kicks the most room inside the legal interval.

Over 100 attempts with a 3° typical spread, that shift raises expected scores from about **41 to 43**. The numerical surprise penalty also falls from about **0.895 to 0.841**; smaller means a legal score is less surprising under the distribution.

This is the bridge to machine learning:

> **A deterministic prediction says what one input produces. A probabilistic model says how plausible many possible outcomes are.**

#### Your turn — cost of poor consistency

Before changing the number, predict what a wider cluster will do:

```python
typical_spread = 8.0  # change to 1.0 after predicting the effect
chance = probability_of_scoring(cluster_centered_aim, typical_spread)
print(f"typical spread={typical_spread}°: expected scores per 100={100 * chance:.1f}")
print(f"surprise penalty={-np.log(chance + 1e-12):.3f}")
```

This calculation does not claim every football error follows a perfect bell curve. It shows how an assumption about uncertainty becomes a testable probability.

<details>
<summary><strong>Connect the surprise penalty to classification</strong></summary>

The formal name for the surprise penalty is **negative log-likelihood**. In classification, cross-entropy is the negative log probability assigned to the correct class.

The Gaussian model here and the softmax probabilities used for classification are different distributions, but both use the same principle: **assign high probability to outcomes like the ones observed**.

</details>

---

## Part 5 — From a Local Derivative to Gradient Descent

The football derivative answered:

> “Given the motion now, where is the next nearby point?”

Challenger Deep adds an adjustable thrust control. Its derivative answers:

> “If thrust changed a tiny amount at this depth, would the mission plan become better or worse?”

The probe must reach roughly **10,900 metres** while balancing:

- **distance:** close the remaining gap;
- **time:** avoid exhausting the underwater window;
- **pressure:** avoid unnecessarily aggressive thrust in deep water.

The mission computer needs one ruler for comparing plans:

> **mission score = distance penalty + time penalty + pressure penalty**

Lower is better. A plan can improve one concern while worsening another, but the total score records the trade-off.

Name the current depth $d$, the adjustable control $u$, and the score $L$:

$$L(d,u)=\text{distance penalty}+\text{time penalty}+\text{pressure penalty}$$

During one derivative test, the physical probe does not move, so depth $d$ is frozen. We vary only $u$. At that fixed depth, the score behaves like a one-variable curve of control versus loss.

<details>
<summary><strong>Audit the exact illustrative mission score used by the animations and game</strong></summary>

Raw control can be any real number, but physical thrust must stay between `0` and `1`. The simulator converts raw control $u$ into bounded thrust fraction $s$ with a sigmoid:

$$s=\frac{1}{1+e^{-u}}$$

At current depth $d$:

1. water conditions reduce effective speed:
   $$v=130\,s\left(1-0.32\left(\frac{d}{10900}\right)^{1.7}\right)$$
2. one turn predicts ten minutes of descent:
   $$d_{next}=\min(10900,\ d+10v)$$
3. remaining distance is $r=10900-d_{next}$;
4. predicted pressure is $p=0.01005\,d_{next}$ MPa.

The score normalizes and weights the three concerns:

$$L(d,u)=\left(\frac{r}{10900}\right)^2+0.14\left(\frac{r/v}{190}\right)^2+0.10\left(\frac{p}{115}\right)^4s^2$$

Read the terms from left to right:

- fraction of target distance still remaining;
- estimated remaining time compared with the 190-minute window;
- pressure exposure, amplified in deep water and by aggressive thrust.

These weights are illustrative design choices, not universal ocean-engineering constants.

The formula reproduces the values used later:

- surface state: $L(0,0.15)\approx0.9586$;
- deep state: $L(8600,1.22)\approx0.0477$.

</details>

Gradient descent changes $u$. It does not change depth directly. After the parameter update is checked, the revised control produces one physical probe step.

### Derivation 1 — Discover the Sign from Two Tiny Tests

First separate the two kinds of movement:

![Diagram separating a gradient step in parameter space from the probe's physical step through ocean depth](images/challenger-gradient-two-spaces.svg)

The physical probe has not moved. Hold its current depth fixed and vary only thrust control.

To make that frozen-depth view explicit, write:

$$L_d(u)=L(d,u)$$

Read $L_d(u)$ as:

> **mission score as a function of control $u$, while depth $d$ stays fixed**

![Animation deriving the local derivative sign by comparing tiny less-thrust and more-thrust tests at the surface and in deep water](images/challenger-derivative-sign-test.gif)

At the surface, the animation reports:

- current score: `0.9586`;
- score after a tiny move left: `0.9760`;
- score after a tiny move right: `0.9430`.

Moving right lowered the score. Therefore the frozen-depth score curve falls from left to right, so its derivative is negative.

Near 8,600 m the comparison reverses:

- tiny move left: score `0.0466`;
- current score: `0.0477`;
- tiny move right: score `0.0487`.

Now moving left lowers the score, so the derivative is positive.

### Turn the Comparison into a Rate

The test used two controls separated by `0.24`: one `0.12` left of NOW and one `0.12` right.

At the surface:

$$\frac{\text{score change}}{\text{control change}}\approx\frac{0.9430-0.9760}{0.24}\approx-0.1375$$

The negative result matches the animation.

Now replace the concrete test distance `0.12` with the symbol $\varepsilon$:

- $u-\varepsilon$ means a tiny move left;
- $u+\varepsilon$ means a tiny move right;
- the full tested distance is $2\varepsilon$.

The nearby slope estimate becomes:

$$\frac{dL_d}{du}\approx\frac{L_d(u+\varepsilon)-L_d(u-\varepsilon)}{2\varepsilon}$$

As the test distance shrinks, this estimate approaches the local derivative at NOW.

> **The derivative sign identifies uphill. Gradient descent chooses the opposite direction.**

### Derivation 2 — Direction Is Not Enough; Choose a Step Size

At 8,600 m, the frozen-depth score curve has local derivative about `+0.0088`.

Positive means moving thrust right raises the score. The downhill direction is therefore left.

But **how far left?** The derivative is local advice, not permission to jump anywhere on the curve.

![Animation comparing a tiny, useful, and oversized step taken along the same downhill derivative direction](images/challenger-step-size-comparison.gif)

### A Deliberately Exaggerated Step-Size Experiment

To make step-size behavior visible, the animation temporarily removes the game's safety cap and compares three large scale values on the same local curve:

| Control move | New score | Interpretation |
| ---: | ---: | --- |
| `−0.088` | `0.0469` | Improves, but barely moves. |
| `−1.059` | `0.0415` | Lands near the local valley. |
| `−2.118` | `0.0547` | Crosses the valley and becomes worse. |

These are teaching-scale moves, not the game's operating settings. All three use the same downhill direction; only their scale differs.

Call the scale the **learning rate** and name it $\eta$ (eta).

The verbal rule is:

> **raw control change = − learning rate × local derivative**

Using the animation's middle case:

$$\text{raw control change}\approx-120\times0.0088\approx-1.06$$

Now shorten the names:

- $\Delta u_{raw}$ = requested control change;
- $dL_d/du$ = local derivative at fixed depth;
- $\eta$ = learning rate.

Therefore:

$$\Delta u_{raw}=-\eta\frac{dL_d}{du}$$

### The Actual Game Rule

The game uses learning rate `3`, then clips the requested change into the safe interval from `−0.18` to `+0.18`:

$$\Delta u_{applied}=\operatorname{clip}\left(-3\frac{dL_d}{du},-0.18,+0.18\right)$$

At the surface, the derivative is about `−0.1372`:

1. raw request: $-3\times(-0.1372)=+0.4116$;
2. safety cap: `+0.4116` becomes `+0.18`;
3. new control: $u_{new}=u+0.18$.

Near 8,600 m, the derivative is only `+0.0088`:

1. raw request: $-3\times0.0088=-0.0264$;
2. this is already inside the cap;
3. new control: $u_{new}=u-0.0264$.

The generic update remains:

$$u_{new}=u+\Delta u_{applied}$$

### Derivation 3 — Recalculate After Every Physical Step

The update does not finish the mission. It chooses the control for one short physical move.

![Close-up gradient-descent animation showing one local control-and-score rectangle before each physical probe move](images/challenger-gradient-step-closeup.gif)

Each turn repeats:

1. measure the local derivative;
2. choose the negative-gradient direction;
3. scale it into a bounded control step;
4. confirm that the predicted score falls;
5. move the probe for ten minutes;
6. rebuild the score curve at the new depth.

The close-up shows why remeasurement matters. Early derivatives are negative, so increasing thrust helps. Near the bottom they become positive, so the improving direction reverses and thrust decreases.

> **Gradient descent is not “keep moving the same way.” It is “measure here, move a little downhill, then measure again.”**

### Concept Check

| Question | Answer |
| --- | --- |
| What does the derivative provide? | The local uphill direction and steepness. |
| Why subtract it? | Subtraction points the update downhill. |
| What does the learning rate provide? | The step size. |
| Why cap the move? | Local slope information should not authorize a reckless jump. |
| Why recompute after the probe moves? | The mission score changes with depth, pressure, and remaining time. |

<details>
<summary><strong>Optional practice: pilot the probe yourself</strong></summary>

The notebook has already derived the concepts. The external game is a practice environment where you can follow or deliberately fight the derivative.

<a href="challenger-deep-gradient-game.html" target="_blank"><strong>Open the optional Challenger Deep gradient game</strong></a>

Suggested experiment:

1. Follow the negative derivative once and observe the score fall.
2. Reset and choose the opposite direction once; observe the score rise.
3. Reset and complete the 14-turn mission.

</details>

The football and probe now answer two related local-change questions:

- **Football:** how is position changing right now?
- **Probe:** how would mission score change if thrust moved slightly?

ML Basics comes next and applies repeated downhill updates to fitted models.

→ **Next:** [`../01-ml-basics/ml-basics.ipynb`](../01-ml-basics/ml-basics.ipynb) — use local gradients to fit regression and classification models.